# VDCP historical EfficientNet-B5 + FPN — T4

Private course experiment. Fixed 8,797 training / 1,885 validation images; no test data.
Single GPU, actual batch 18 (16 only if OOM), original single-channel normalization,
cat FPN, global BCE+Dice. First checkpoint review at epoch 60; each session has a
10.5-hour budget. Original pseudo-labeling and ensemble are excluded.


In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, subprocess, sys, zipfile
import torch, cv2, numpy as np, scipy
print({"python":platform.python_version(), "torch":torch.__version__,
       "numpy":np.__version__, "opencv":cv2.__version__, "scipy":scipy.__version__,
       "cuda":torch.cuda.is_available(), "gpus":torch.cuda.device_count()})
assert torch.cuda.is_available(), "Choose GPU T4 x2 in Notebook settings first"
print(torch.cuda.get_device_name(0))
# Use one T4. Device memory is not pooled across the two GPUs.


In [ ]:
candidates = [p for p in Path('/kaggle/input').rglob('configs/fpn_v9_kaggle_original.json')
              if 'input_extracted' not in p.parts]
if not candidates:
    archives=list(Path('/kaggle/input').rglob('vdcp_b5_private.zip'))
    assert len(archives)==1, "Attach the private VDCP B5 dataset"
    destination=Path('/kaggle/working/input_extracted')
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        for item in z.infolist():
            target=(destination/item.filename).resolve()
            assert target.is_relative_to(destination.resolve()), "Unsafe archive path"
        z.extractall(destination)
    candidates=list(destination.rglob('configs/fpn_v9_kaggle_original.json'))
assert len(candidates)==1, "Require exactly one VDCP source bundle"
ROOT=candidates[0].parents[1]
manifest=json.loads((ROOT/'bundle_manifest.json').read_text())
assert (manifest['train_images'],manifest['validation_images'],manifest['test_images_included'])==(8797,1885,0)
for relative,record in manifest['files'].items():
    path=(ROOT/relative).resolve()
    assert path.is_relative_to(ROOT.resolve()) and path.stat().st_size==record['bytes']
    digest=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):
            digest.update(chunk)
    assert digest.hexdigest()==record['sha256'], relative
print('Verified source, ImageNet weights and all',len(manifest['files']),'files; ROOT=',ROOT)
OUTPUT=Path('/kaggle/working/kaggle_b5')
OUTPUT.mkdir(exist_ok=True)


In [ ]:
# First stage. Change to 100/150/200 only after reviewing the 60-epoch results.
STOP_AT=60
SESSION_HOURS=10.5
resume_candidates=list(Path('/kaggle/input').rglob('kaggle_b5/last.pt'))
assert len(resume_candidates)<=1, "Attach only one previous run output"
resume=None
if (OUTPUT/'last.pt').exists():
    # A live rerun must never replace newer progress with an older attachment.
    resume=OUTPUT/'last.pt'
elif resume_candidates:
    resume=resume_candidates[0]
    # Preserve all preceding best models, history and profiler evidence.
    for file in resume.parent.iterdir():
        if file.is_file():
            shutil.copyfile(file,OUTPUT/file.name)
    resume=OUTPUT/'last.pt'
command=[sys.executable,str(ROOT/'scripts/train_kaggle_b5.py'),
         '--root',str(ROOT),'--output',str(OUTPUT),'--stop-at',str(STOP_AT),
         '--session-hours',str(SESSION_HOURS)]
if resume:
    command+=['--resume',str(resume)]
env=os.environ.copy()
env['PYTHONUNBUFFERED']='1'
with (OUTPUT/'training.log').open('a',encoding='utf-8') as log:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,bufsize=1,env=env)
    for line in process.stdout:
        print(line,end='',flush=True); log.write(line); log.flush()
    result=process.wait()
assert result==0, f'Training exited with code {result}; see training.log'
print((OUTPUT/'status.json').read_text())


## Resume / evaluate

Save & Run All persists `/kaggle/working/kaggle_b5` when the session completes.
If the time budget is reached before epoch 60, attach this Notebook's last version
as an input to the next session. The same source/config/split/weights identity is
required. `last.pt` restores optimizer, scheduler, scaler and RNG at a completed
epoch. `best_region_f2.pt` and `best_dice.pt` are separate selection criteria.

Final calibrated course evaluation is a separate run of the packaged script with
`--evaluate-only --resume /path/to/best_region_f2.pt`; training here reports full
validation Dice and the fixed-protocol regional metrics every five epochs.
Dice on this validation split is not the competition leaderboard score.
